# Laboratorio 4: Detección de Movimiento y Seguimiento de Objetos

### Objetivo
En esta práctica, aprenderás a implementar algoritmos de detección de movimiento mediante sustracción de fondo y flujo óptico, y explorarás el filtro de Kalman para el seguimiento de objetos.

## Materiales
- **Python 3.8+**
- **OpenCV**: Puedes instalarlo con `pip install opencv-python`
- **Dataset de video**: Se usará un archivo de video o la cámara en tiempo real para probar los métodos de detección de movimiento.

In [2]:
import cv2
import os
import numpy as np
import time

## Apartado C: Sustracción de Fondo

### Tarea C.1: Carga de Video
Carga un video en el cual se detectarán objetos en movimiento. Puedes utilizar
un video local o la cámara en tiempo real.

In [14]:
#TODO: Use VideoCapture from OpenCV to read the video (visiontraffic.avi)
videopath = "../data/partC/visiontraffic.avi"  # Path to the video file
cap = cv2.VideoCapture(filename=videopath)  # Complete this line to read the video file

#TODO: Check if the video was successfully opened
if not cap.isOpened():  # Comprobamos que se abrió bien el video
    print('Error: Could not open the video file')
    os._exit(0)

#TODO: Get the szie of frames and the frame rate of the video
frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
frame_rate = cap.get(cv2.CAP_PROP_FPS)

# Añadimos un print para mostrar las características del video con el que estamos trabajando
print("Video properties:")
print(" - Width :", frame_width)
print(" - Height:", frame_height)
print(" - FPS   :", frame_rate)

#TODO: Use a loop to read the frames of the video and store them in a list
frames = []
while True:
    ret, frame = cap.read() # Esto me permite leer cada uno de los frames que componen el video (al final el vídeo es una sucesión de imágenes)
    if not ret:
        break
    frames.append(frame)

cap.release()

print(f"\nTotal frames read: {len(frames)}")

Video properties:
 - Width : 640
 - Height: 360
 - FPS   : 29.97002997002997

Total frames read: 531


In [ ]:
# Para ver el video cargado
cap = cv2.VideoCapture(videopath)

delay = int(1000 / frame_rate)   # delay en ms

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break
    
    cv2.imshow("Video tras sustraccion", frame)
    if cv2.waitKey(delay) & 0xFF == ord('q'):  # Pulsar 'q' para salir
        break

cap.release()
cv2.destroyAllWindows()

### Tarea C.2: Sustración de Fondo mediante diferencia de frames
Realiza una sustracción de fondo mediante diferencia de frames, para ello guarda
un frame con el fondo estático y úsalo como frame de referencia de fondo.

In [4]:
#TODO:  Show the frames to select the reference frame, press 'n' to move to the next frame and 's' to select the frame
for i, frame in enumerate(frames):
    # Show the frame
    cv2.imshow(f'Video frame {i}', frame)
    # Wait for the key
    key = cv2.waitKey(0)
    # If the key is 'n' continue to the next frame
    if key == ord('n'):
        continue
    # If the key is 's' select the frame as the reference frame
    elif key == ord('s'):
        # Copy the frame to use it as a reference
        reference_frame = frame
        # Convert the reference frame to grayscale
        reference_frame = cv2.cvtColor(reference_frame, cv2.COLOR_BGR2GRAY)
        print('Frame {} selected as reference frame'.format(i))
        break

cv2.destroyAllWindows()

Frame 0 selected as reference frame


In [5]:
#TODO: Compute the difference between the reference frame and the rest of the frames and show the difference
for frame in frames:
    # time.sleep(0.001)  # Este sleep es por si queremos ver el vídeo más lentamente y fijarnos bien 
    # Convert the frame to grayscale
    frame = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    diff = cv2.absdiff(reference_frame, frame)
    cv2.imshow('Diferencia', diff)
    key = cv2.waitKey(1)
    if key == ord('q'):
        break

cv2.destroyAllWindows()

### Tarea AC.3: Configuración de la sustración de fondo con GMM
Configura el sustractor de fondo usando el modelo de mezcla de gaussianas
adaptativas (MOG2).

In [6]:
#TODO: Use MOG2 to detect the moving objects in the video

history = 400  # Number of frames to use to build the background model (valores grandes: más estable pero tarda en adaptarse, valores pequeños: se adapta rápido pero es inestable)
varThreshold = 20  # Threshold to detect the background (umbral que determina si un pixel es fondo o no lo es)
detectShadows = True  # If True the algorithm detects the shadows

# Create the MOG2 object
mog2 = cv2.createBackgroundSubtractorMOG2(history, varThreshold, detectShadows)


### Tarea C.3: Aplicación de la Sustracción de Fondo

Aplica la sustracción de fondo en cada frame para extraer los objetos en movimiento.

In [17]:
#TODO: Use a loop to detect the moving objects in the video using the MOG2 algorithm and 
# save a video storing the parameters at the name of the file

# Create a folder to store the videos
output_folder = "../data/results"
if not os.path.exists(output_folder):
    os.makedirs(output_folder)

videoname = f'output_{history}_{varThreshold}_{detectShadows}.avi' # Name of the output video file with the parameters
videopath = os.path.join(output_folder, videoname)

# Create a VideoWriter object to save the video
fourcc = cv2.VideoWriter_fourcc(*"XVID") # Codec to use
frame_size = (frame_width, frame_height) # Size of the frames
fps = frame_rate # Frame rate of the video
out = cv2.VideoWriter(filename=videopath, fourcc=fourcc, fps=fps, frameSize=frame_size)

for frame in frames:
    # Apply the MOG2 algorithm to detect the moving objects
    mask = mog2.apply(frame)
    # Convert to BGR the mask to store it in the video
    mask = cv2.cvtColor(mask, cv2.COLOR_GRAY2BGR)
    # Save the mask in a video
    out.write(mask)

out.release()
print(f"Vídeo guardado en: {videopath}")

Vídeo guardado en: ../data/results\output_400_20_True.avi


In [18]:
# Para ver el video resultado
cap = cv2.VideoCapture(videopath)

delay = int(1000 / frame_rate)   # delay en ms

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break
    
    cv2.imshow("Video tras sustraccion", frame)
    if cv2.waitKey(delay) & 0xFF == ord('q'):  # Pulsar 'q' para salir
        break

cap.release()
cv2.destroyAllWindows()


**Preguntas del Apartado C**
1. ¿Cómo afecta la variable `varThreshold` a la precisión de la detección?
2. ¿Qué ventajas presenta `createBackgroundSubtractorMOG2` frente a métodos simples de diferencia de imágenes?

### Pregunta C1:

Vamos a generar distintos videos de detección para comprobar visualmente cómo afecta la modificación del parámetro varThreshold, a la hora de tener mayor o menor precisión de detección. Finalmente, analizaremos los resultados obtenidos y comentaremos lo observado.

In [24]:
# En el mismo bucle creamos el MOG2, guardamos el video, y luego visualizamos el resultado

# Carpeta de salida 
output_folder = "../data/PartC/diferent_varThresholds"
os.makedirs(output_folder, exist_ok=True)

# Parámetros que no cambiamos
history = 400          # mantenemos history fijo
detectShadows = True   # y la detección de sombras también

# Valores de varThreshold que sí vamos cambiando
varThreshold_values = [5, 15, 25, 40, 60]

delay = int(1000 / frame_rate)   # delay en ms

for vt in varThreshold_values:
    print(f"Generando vídeo para varThreshold = {vt}")

    # Crear el sustractor MOG2 con este varThreshold
    mog2 = cv2.createBackgroundSubtractorMOG2(
        history=history,
        varThreshold=vt,
        detectShadows=detectShadows
    )

    # Nombre del vídeo de salida
    videoname = f"output_h{history}_vt{vt}_sh{detectShadows}.avi"
    videopath = os.path.join(output_folder, videoname)

    # VideoWriter
    fourcc = cv2.VideoWriter_fourcc(*"XVID")
    frame_size = (frame_width, frame_height)
    fps = frame_rate
    out = cv2.VideoWriter(videopath, fourcc, fps, frame_size)

    for frame in frames:
        # Máscara de movimiento
        mask = mog2.apply(frame)

        # Pasar a BGR para poder guardarlo en el vídeo
        mask_bgr = cv2.cvtColor(mask, cv2.COLOR_GRAY2BGR)

        out.write(mask_bgr)

    out.release()
    print(f"Vídeo guardado en: {videopath}")

    # Para ver el video resultado
    cap = cv2.VideoCapture(videopath)

    delay = int(1000 / frame_rate)   # delay en ms

    print(f"Reproduciendo vídeo: {videopath}\n")

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
        
        cv2.imshow(f"Video Sustraccion : Threshold = {vt}", frame)
        if cv2.waitKey(delay) & 0xFF == ord('q'):  # Pulsar 'q' para salir
            break

    cap.release()
    cv2.destroyAllWindows()


Generando vídeo para varThreshold = 5
Vídeo guardado en: ../data/PartC/diferent_varThresholds\output_h400_vt5_shTrue.avi
Reproduciendo vídeo: ../data/PartC/diferent_varThresholds\output_h400_vt5_shTrue.avi

Generando vídeo para varThreshold = 15
Vídeo guardado en: ../data/PartC/diferent_varThresholds\output_h400_vt15_shTrue.avi
Reproduciendo vídeo: ../data/PartC/diferent_varThresholds\output_h400_vt15_shTrue.avi

Generando vídeo para varThreshold = 25
Vídeo guardado en: ../data/PartC/diferent_varThresholds\output_h400_vt25_shTrue.avi
Reproduciendo vídeo: ../data/PartC/diferent_varThresholds\output_h400_vt25_shTrue.avi

Generando vídeo para varThreshold = 40
Vídeo guardado en: ../data/PartC/diferent_varThresholds\output_h400_vt40_shTrue.avi
Reproduciendo vídeo: ../data/PartC/diferent_varThresholds\output_h400_vt40_shTrue.avi

Generando vídeo para varThreshold = 60
Vídeo guardado en: ../data/PartC/diferent_varThresholds\output_h400_vt60_shTrue.avi
Reproduciendo vídeo: ../data/PartC/difer

### Observaciones:

La variable varThreshold controla el umbral de decisión que usa MOG2 para clasificar un píxel como fondo o primer plano en función de la distancia respecto al modelo gaussiano del fondo.

Con valores bajos (p.ej. 5) el algoritmo es muy sensible: cualquier pequeña variación se detecta como movimiento. En los vídeos correspondientes se observa mucha detección espuria en el asfalto y en zonas estáticas (ruido y falsas alarmas).

Con valores intermedios (15–25) el resultado es más equilibrado: se detectan de forma bastante completa los vehículos y se reduce el ruido del fondo. Visualmente la segmentación es más estable y las siluetas están mejor definidas.

Con valores altos (40–60) el modelo se vuelve demasiado conservador: el fondo es muy estable pero algunos vehículos aparecen parcialmente segmentados o incluso desaparecen, lo que implica pérdida de detecciones (falsos negativos).

Por tanto, varThreshold afecta directamente al compromiso entre sensibilidad y precisión de la detección: valores bajos aumentan la sensibilidad pero introducen ruido, mientras que valores altos reducen las falsas detecciones a costa de perder objetos reales. En nuestro vídeo de tráfico, un valor intermedio alrededor de XX (el que tú veas mejor en tus resultados) ofrece el mejor equilibrio entre ambos efectos.

### Pregunta C2:

El método createBackgroundSubtractorMOG2 presenta varias ventajas significativas frente a los métodos simples basados en la diferencia de imágenes, especialmente en entornos reales como el vídeo de tráfico empleado en la práctica. Mientras que la diferencia directa entre un frame de referencia y el frame actual solo permite detectar cambios brutos en la imagen, MOG2 utiliza un modelo estadístico mucho más robusto y adaptable.

En primer lugar, MOG2 modela cada píxel mediante una mezcla de gaussianas, lo que le permite distinguir entre variaciones normales del fondo (como sombras, ligeras variaciones de iluminación o movimiento del propio entorno) y objetos realmente significativos en movimiento. Esto contrasta con la diferencia de imágenes, que considera como movimiento cualquier cambio de intensidad, generando detecciones erróneas incluso ante ruidos mínimos.

En segundo lugar, MOG2 actualiza dinámicamente el modelo del fondo, lo que permite que el sistema se adapte cuando cambian las condiciones de la escena, por ejemplo, la transición del día a la tarde, cambios de sombras, o la acumulación progresiva de coches en el tráfico. El método de diferencia de imágenes es estático: depende completamente del frame de referencia elegido y no incorpora ningún mecanismo de actualización.

Otro aspecto importante es que MOG2 incluye un sistema integrado de detección de sombras, permitiendo diferenciarlas de los objetos reales mediante una codificación específica de la máscara. En contraste, la diferencia simple no puede identificar sombras y tiende a clasificarlas como movimiento, lo que reduce significativamente la precisión.

Finalmente, MOG2 introduce parámetros como history, varThreshold y detectShadows, que permiten ajustar el comportamiento del modelo en función del tipo de escena. Esto lo convierte en un método flexible y configurable, mientras que la diferencia de imágenes únicamente ofrece un resultado binario y poco controlable.

En conjunto, estas características hacen que createBackgroundSubtractorMOG2 proporcione una detección más precisa, estable y robusta, siendo especialmente adecuado para secuencias de vídeo complejas como la empleada en la práctica, donde las técnicas simples de diferencia presentan grandes limitaciones.

## Apartado D: Flujo Óptico

### Tarea D.1: Configuración del Flujo Óptico

Consulta la documentación de cv2.calcOpticalFlowPyrLK para ver que parametros se deben definir para realizar el calculo del flujo óptico

In [ ]:
#TODO: Use VideoCapture from OpenCV to read the video (slow_traffic_small.mp4)
videopath = None  # Path to the video file
cap = cv2.VideoCapture(None)  # Complete this line to read the video file

#TODO: Check if the video was successfully opened
if ...:
    print('Error: Could not open the video file')
    os._exit(0)

#TODO: Get the szie of frames and the frame rate of the video
frame_width = None
frame_height = None
frame_rate = None

#TODO: Use a loop to read the frames of the video and store them in a list
frames = []
while True:
    ret, frame = None
    if not ret:
        break
    frames.append(frame)


cap.release()

# Define the parameters for Lucas-Kanade optical flow
winSize=(None, None)
maxLevel=None
criteria= (cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT, 10, 0.03)

### Tarea D.2: Detección de Puntos de Interés

Detecta los puntos de interés iniciales usando el algoritmo de Shi-Tomasi (cv2.goodFeaturesToTrack) en el primer frame.

In [ ]:
#TODO: Detect the initial points of interest in the first frame

# Convert the first frame to grayscale
prev_gray = cv2.cvtColor(None, None)

# Define the parameters of the Shi-Tomasi algorithm
mask = None
maxCorners = None
qualityLevel = None
minDistance = None
blockSize = None

# Use the function goodFeaturesToTrack to detect the points of interest
p0 = cv2.goodFeaturesToTrack(prev_gray, mask=mask, maxCorners=maxCorners, qualityLevel=qualityLevel, minDistance=minDistance, blockSize=blockSize)

### Tarea D.3: Cálculo y Visualización del Flujo Óptico

In [ ]:
#TODO: Use a loop to track the points of interest in the rest of the frames

# Create a mask image for drawing purposes
mask = np.zeros_like(frame)

for i, frame in enumerate(frames[1:]):
    input_frame = None
    # Convert the frame to grayscale
    frame_gray = cv2.cvtColor(input_frame, cv2.COLOR_BGR2GRAY)
    # Calculate the optical flow using the Lucas-Kanade algorithm

    p1, st, err = cv2.calcOpticalFlowPyrLK(None, None, None, None)

    # Select the points that were successfully tracked
    good_new = p1[st == 1]
    good_old = p0[st == 1]

    # Draw the tracks
    for i, (new, old) in enumerate(zip(good_new, good_old)):
        a, b = new.ravel().astype(int)
        c, d = old.ravel().astype(int)
        input_frame = cv2.circle(input_frame, (a, b), 5, (0, 0, 255), -1)
        mask = cv2.line(mask, (a, b), (c, d), (0, 255, 0), 2)

    # Update the inputs for the next iteration
    prev_gray = None
    p0 = good_new.reshape(-1, 1, 2)
    
    # Show the frame with the tracks
    cv2.imshow('Frame', cv2.add(input_frame, mask))
    key = cv2.waitKey(1)
    if key == ord('q'):
        break

cv2.destroyAllWindows()

**Preguntas del Apartado D**
1. ¿Qué efecto tiene el parámetro `winSize` en la precisión del flujo óptico?
2. ¿Cómo influye el parámetro `qualityLevel` en la función `cv2.goodFeaturesToTrack` al detectar puntos de interés?

## Apartado E: Filtro de Kalman para Seguimiento de Objetos

### Tarea E.1: Configuración del Filtro de Kalman

Inicializa el filtro de Kalman (cv2.KalmanFilter) con una matriz de medición y transición adecuada para un seguimiento en dos dimensiones.

In [ ]:
#TODO: Use VideoCapture from OpenCV to read the video (slow_traffic_small.mp4)
videopath = None  # Path to the video file
cap = cv2.VideoCapture(None)  # Complete this line to read the video file

#TODO: Check if the video was successfully opened
if ...:
    print('Error: Could not open the video file')

#TODO: Get the szie of frames and the frame rate of the video
frame_width = None
frame_height = None
frame_rate = None

#TODO: Use a loop to read the frames of the video and store them in a list
frames = []
while True:
    ret, frame = None
    if not ret:
        break
    frames.append(frame)


cap.release()


# TODO: Create the Kalman filter object
kf = cv2.KalmanFilter(None, None)
# TODO: Initialize the state of the Kalman filter
kf.measurementMatrix = None # Measurement matrix np.array of shape (2, 4) and type np.float32
kf.transitionMatrix = None # Transition matrix np.array of shape (4, 4) and type np.float32
kf.processNoiseCov = None # Process noise covariance np.array of shape (4, 4) and type np.float32

measurement = np.array((2, 1), np.float32)
prediction = np.zeros((2, 1), np.float32)

#TODO: Show the frames to select the initial position of the object

for i, frame in enumerate(frames):
    # Show the frame
    cv2.imshow('Frame', frame)
    # Wait for the key
    key = cv2.waitKey(0)
    # If the key is 'n' continue to the next frame
    if key == ord('n'):
        continue
    # If the key is 's' select the position of the object
    elif key == ord('s'):
        # Select the position of the object
        x, y, w, h = cv2.selectROI('Frame', frame, False)
        track_window = (x, y, w, h)
        #TODO: Compute the center of the object
        cx = None
        cy = None
        #TODO: Initialize the state of the Kalman filter
        kf.statePost = np.array([[None], [None], [0], [0]], np.float32)
        # Initialize the covariance matrix
        kf.errorCovPost = np.eye(4, dtype=np.float32)
        #Predict the position of the object
        prediction = kf.predict()

        #TODO: Update the measurement and correct the Kalman filter
        measurement = np.array([[None], [None]], np.float32)
        kf.correct(measurement)

        #TODO: Crop the object
        crop = frame[None:None, None:None].copy()
        hsv_crop = cv2.cvtColor(None, None)
        #TODO: Compute the histogram of the cropped object (Reminder: Use only the Hue channel (0-180))
        crop_hist = cv2.calcHist([None], [0], mask=None, histSize=[None], ranges=[None, None])
        cv2.normalize(crop_hist, crop_hist, 0, 255, cv2.NORM_MINMAX)
        
        print(f'Initial position selected: {x}, {y}')
        break

cv2.destroyAllWindows()

Initial position selected: 2, 266


### Tarea E.2: Predicción y Corrección del Estado

Realiza la predicción del estado y corrige la posición estimada en cada iteración.

In [ ]:
#TODO: Use the Kalman filter to predict the position of the points of interest

term_crit = (cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT, 30, 1)

for frame in frames[i:]:
    #TODO: Copy the frame 
    input_frame = None
    #TODO: Convert the frame to HSV
    img_hsv = cv2.cvtColor()
    
    # Compute the back projection of the histogram
    img_bproject = cv2.calcBackProject([img_hsv], [0], crop_hist, [0, 180], 1)
    
    # Apply the mean shift algorithm to the back projection
    ret, track_window = cv2.meanShift(img_bproject, track_window, term_crit)
    x_,y_,w_,h_ = track_window
    #TODO: Compute the center of the object
    c_x = None
    c_y = None
    
    # Predict the position of the object
    prediction = kf.predict()

    #TODO: Update the measurement and correct the Kalman filter
    measurement = np.array([[None], [None]], np.float32)
    kf.correct(measurement)

    
    # Draw the predicted position
    cv2.circle(input_frame, (int(prediction[0][0]), int(prediction[1][0])), 5, (0, 0, 255), -1)
    cv2.circle(input_frame, (int(c_x), int(c_y)), 5, (0, 255, 0), -1)
    
    # Show the frame with the predicted position
    cv2.imshow('Frame', input_frame)
    key = cv2.waitKey(0)
    if key == ord('q'):
        break

cv2.destroyAllWindows()


**Preguntas del Apartado E**
1. ¿Cómo afecta el valor de `transitionMatrix` a la predicción en el filtro de Kalman?
2. ¿Cuál es la diferencia entre `measurementMatrix` y `transitionMatrix` en el contexto del seguimiento de objetos?